In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [ ]:
# ==========================================
# FASE A - DATA PIPELINE E AUGMENTATION
# ==========================================

TRAIN_DIR = (r"C:\Users\Valentina\rps")
TEST_DIR = (r"C:\Users\Valentina\rps-test-set")

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
VAL_SPLIT = 0.2

# Training generator con data augmentation
train_gen = ImageDataGenerator(
    rescale=1./255,             # normalizzazione
    rotation_range=40,          # rotazione tra -40° e +40°
    width_shift_range=0.2,      # spostamento casuale in orizzontale fino al 20% della larghezza immagine
    shear_range=0.2,            # inclinazione immagine max 20%
    horizontal_flip=True,       # ribaltamento orizzontale
    fill_mode='nearest',        # evita zone vuote dopo le trasformazioni utilizzando i pixel più vicini al bordo
    validation_split=VAL_SPLIT  # 20% dei dati del training riservati alla validazione
)

# Generator per la VALIDATION: stesso split, ma SENZA augmentation (solo normalizzazione, dati puliti)
val_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VAL_SPLIT
)

# Test generator: solo normalizzazione, nessuna augmentation
test_gen = ImageDataGenerator(rescale=1./255)

train_generator = train_gen.flow_from_directory(    # flow_from_directory() legge le immagini organizzate in sotto_cartelle
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',   # necessario per 3 classi 
    subset='training',
    seed=42                     # riproducibilità
)

validation_generator = val_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42                     # vede le stesse immagini di train_generator divise nello stesso modo
)

test_generator = test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   # utile per analisi della predizione in Fase D
)

print("\nClassi rilevate:", train_generator.class_indices)
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

Found 2016 images belonging to 3 classes.
Found 504 images belonging to 3 classes.
Found 372 images belonging to 3 classes.

Classi rilevate: {'paper': 0, 'rock': 1, 'scissors': 2}
Training samples: 2016
Validation samples: 504
Test samples: 372


In [13]:
from tensorflow.keras import layers, models

In [14]:
# ==========================================
# FASE B - ARCHITETTURA CNN
# ==========================================

model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),              # immagini 150x150, 3 canali di colore (RGB) --> output shape (150x150x3)

    layers.Conv2D(32, (3,3), activation='relu'),    # la rete impara 32 filtri di dim. 3x3 con attivazione relu (Rectified Linear Unit): se x < 0 → 0, se x > 0 → x --> output shape (148x148x32)
    layers.MaxPooling2D(2,2),                       # riduzione dim. feature map alla metà dei valori --> output shape (74x74x32)

    layers.Conv2D(64, (3,3), activation='relu'),    # aumento filtri per imparare caratteristiche più complesse --> output shape (72x72x64)
    layers.MaxPooling2D(2,2),                       # output shape (36x36x64)

    layers.Conv2D(128, (3,3), activation='relu'),   # output shape (34x34x128)
    layers.MaxPooling2D(2,2),                       # output shape (17x17x128)

    layers.Conv2D(128, (3,3), activation='relu'),   # output shape (15x15x128)
    layers.MaxPooling2D(2,2),                       # output shape (7x7x128)

    layers.Flatten(),                               # da (7x7x128=6272) a unico vettore 6272
    layers.Dropout(0.5),                            # utile con 4 blocchi conv per limitare overfitting (disattiva casualmente 50% dei neuroni)
    layers.Dense(512, activation='relu'),           # fully-connected-layer (512 neuroni)
    layers.Dense(3, activation='softmax')           # output finale 3 neuroni per le 3 classi (rock-paper-scissors) e attivzione softmax che trasforma output in probabilità (somma 1)
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,147 (13.18 MB)

 Trainable params: 3,454,147 (13.18 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
from tensorflow.keras.callbacks import EarlyStopping, Callback

In [16]:
# ==========================================
# FASE C - COMPILAZIONE E TRAINING
# ==========================================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',    # classificazione multiclasse
    metrics=['accuracy']                # misura delle predizioni corrette
)

# --- Callback custom: stop se accuracy > 98% ---
class StopAt98(Callback):
    def on_epoch_end(self, epoch, logs=None):                                            # esegue il codice alla fine di ogni epoca
        if logs.get('val_accuracy') is not None and logs.get('val_accuracy') > 0.98:     # logs.get() recupera la val_accuracy per quell'epoca (in validation per valutare capacità di generallizzazione), se > 98% 
            print(f"\nRaggiunta validation accuracy {logs.get('val_accuracy'):.2%} — stop training.")
            self.model.stop_training = True                                              # interrompe model.fit()

# --- EarlyStopping ---
early_stopping = EarlyStopping(             # fermare il training se dopo 3 epoche il modello non sta più migliorando sulla validation
    monitor='val_loss',
    patience=3,
    restore_best_weights=True               # ripristina i pesi dell'epoca migliore, non quelli dell'ultima epoca che potrebbero essere peggiori
)

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=[early_stopping, StopAt98()]
)

Epoch 1/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - accuracy: 0.5283 - loss: 0.9242 - val_accuracy: 0.7956 - val_loss: 0.6034
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.8745 - loss: 0.3476 - val_accuracy: 0.9147 - val_loss: 0.2291
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.9464 - loss: 0.1558 - val_accuracy: 0.9643 - val_loss: 0.1430
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.9568 - loss: 0.1343 - val_accuracy: 0.7361 - val_loss: 0.6186
Epoch 5/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.9678 - loss: 0.0971 - val_accuracy: 0.9365 - val_loss: 0.2118
Epoch 6/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 83s 1s/step - accuracy: 0.9678 - loss: 0.0908 - val_accuracy: 0.9782 - val_loss: 0.1060
Epoch 7/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.9836 - loss: 0.0523 - val_accuracy: 0.9306 - val_loss: 0.1807
Epoch 8/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - accuracy: 0.9802 - loss: 0.0625 - val_accuracy: 0.8452 - val_los

Il minimo di val_loss si ha in epoca 6, quindi grazie all'EarlyStopping dopo 3 epoche consecutive (7,8,9) non migliora più e si stoppa.

La StopAt98 invece non è riuscita a scattare. Ha toccato 0.9782 all'epoca 6.

In [ ]:
from PIL import Image   # Python Imaging Library permette a Python di aprire, modificare, analizzare e salvare immagini
import numpy as np

In [ ]:
# ==========================================
# FASE D - TEST E VALIDAZIONE "WILD"
# ==========================================

# Valutazione sul test set:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Accuracy sul test set CGI: {test_acc:.2%}")


# Test reale con foto scattata
def classifica_foto(percorso_immagine):
    # Stesso preprocessing usato in training/test: resize 150x150 + normalizzazione 1/255
    img = Image.open(percorso_immagine).convert('RGB')      # apre la foto e la converte in RGB
    img = img.resize((150, 150))
    img_array = np.array(img) / 255.0                   
    img_array = np.expand_dims(img_array, axis=0)           # aggiunta dela dimensione batch=1 --> (1,150,150,3)

    pred = model.predict(img_array)
    classi = list(train_generator.class_indices.keys())     # .keys() prende i nomi delle classi e restituisce una lista con list() ['paper', 'rock', 'scissors']
    classe_predetta = classi[np.argmax(pred)]               # np.argmax() cerca il valore più grande della predizione e restituisce il nome della classe
    probabilita = np.max(pred)                              # prende il valore più alto della probabilità

    print(f"Predizione: {classe_predetta} (probabilità: {probabilita:.2%})")
    return classe_predetta, probabilita

# Analisi del Domain Gap
classifica_foto(r"C:\Users\Valentina\rps-test-set\mano_vale_1.jpg")
classifica_foto(r"C:\Users\Valentina\rps-test-set\mano_vale_2.jpg")
classifica_foto(r"C:\Users\Valentina\rps-test-set\mano_vale_3.jpg")

12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 301ms/step - accuracy: 0.9570 - loss: 0.1523
Accuracy sul test set CGI: 95.70%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Predizione: rock (probabilità: 87.04%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
Predizione: paper (probabilità: 52.32%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Predizione: paper (probabilità: 85.63%)


('paper', np.float32(0.8563245))

ANALISI:

95.70% di accuratezza sul test set CGI vs errore su foto reale con confusione tra scissors→paper, attribuibile a  differenze (sfondo, posa, texture) rispetto al set fornito